In [ ]:
from datasets import load_dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
target_dir = '../data/samples'
# --- Charger ---
df = pd.read_parquet(target_dir +"/financial_reports_10000.parquet")
print(f"Shape: {df.shape}")

Shape: (10000, 19)


In [6]:
print(df.dtypes)

cik                     object
sentence                object
section                  int64
labels                  object
filingDate              object
name                    object
docID                   object
sentenceID              object
sentenceCount            int64
tickers                 object
exchanges               object
entityType              object
sic                     object
stateOfIncorporation    object
tickerCount              int64
acceptanceDateTime      object
form                    object
reportDate              object
returns                 object
dtype: object


In [7]:
print(df.head(2))

          cik                                           sentence  section  \
0  0000001750  ITEM 1.BUSINESS General AAR CORP. and its subs...        0   
1  0000001750  AAR was founded in 1951, organized in 1955 and...        0   

                         labels  filingDate      name                 docID  \
0  {'1d': 0, '5d': 1, '30d': 0}  2020-07-21  AAR CORP  0000001750_10-K_2020   
1  {'1d': 0, '5d': 1, '30d': 0}  2020-07-21  AAR CORP  0000001750_10-K_2020   

                         sentenceID  sentenceCount tickers exchanges  \
0  0000001750_10-K_2020_section_1_0              1   [AIR]    [NYSE]   
1  0000001750_10-K_2020_section_1_1              2   [AIR]    [NYSE]   

  entityType   sic stateOfIncorporation  tickerCount  \
0  operating  3720                   DE            1   
1  operating  3720                   DE            1   

         acceptanceDateTime  form  reportDate  \
0  2020-07-21T17:19:15.000Z  10-K  2020-05-31   
1  2020-07-21T17:19:15.000Z  10-K  2020-05-31 

In [9]:
# Distribution des labels
df["labels"].value_counts()

labels
{'1d': 1, '5d': 1, '30d': 1}    5289
{'1d': 0, '5d': 1, '30d': 0}    2513
{'1d': 0, '5d': 0, '30d': 1}    1147
{'1d': 1, '5d': 1, '30d': 0}    1051
Name: count, dtype: int64

In [17]:
# Extraire les 3 horizons
df["label_1d"]  = df["labels"].apply(lambda x: x["1d"])
df["label_5d"]  = df["labels"].apply(lambda x: x["5d"])
df["label_30d"] = df["labels"].apply(lambda x: x["30d"])

# Cohérence : si 1d et 5d sont négatifs, 30d l'est-il aussi ?
print(pd.crosstab(
    (df["label_1d"] == 0) & (df["label_5d"] == 0),
    df["label_30d"] == 0
))   

label_30d  False  True 
row_0                  
False       5289   3564
True        1147      0


In [14]:
# Équilibrage
print(f"Ratio positif: {df['labels'].value_counts().mean(axis=0):.2%}")

Ratio positif: 250000.00%


In [15]:
plt.figure(figsize=(6, 4))
sns.countplot(x="labels", data=df)
plt.title("Distribution des labels (0=positive, 1=negative)")
plt.tight_layout()
plt.show()   

TypeError: unhashable type: 'dict'

<Figure size 600x400 with 0 Axes>

In [16]:
df["text_len"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["word_count"], bins=50, ax=axes[0])
axes[0].set_title("Nombre de mots par sentence")
sns.histplot(df["word_count"], hue="label", bins=50, ax=axes[1])
axes[1].set_title("Par label")
plt.tight_layout()
plt.show()

df["word_count"].describe()   

KeyError: 'text'

In [ ]:
# section est un int 0-19 → mapper
section_map = {
    0: "1", 1: "10", 2: "11", 3: "12", 4: "13",
    5: "14", 6: "15", 7: "1A", 8: "1B", 9: "2",
    10: "3", 11: "4", 12: "5", 13: "6", 14: "7",
    15: "7A", 16: "8", 17: "9", 18: "9A", 19: "9B"
}
df["section_name"] = df["section"].map(section_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(y="section_name", data=df, order=df["section_name"].value_counts().index, ax=axes[0])
axes[0].set_title("Fréquence par section")

# Proportion de label=1 (négatif) par section
pct = df.groupby("section_name")["label"].mean().sort_values(ascending=False)
pct.plot(kind="barh", ax=axes[1], color="red", alpha=0.7)
axes[1].set_title("Taux de réaction négative par section")
plt.tight_layout()
plt.show()   

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(ngram_range=(2, 2), max_features=10000, stop_words="english")
cv.fit(df["text"])
features = cv.get_feature_names_out()

pos = cv.transform(df.loc[df["label"] == 0, "text"]).sum(axis=0).A1
neg = cv.transform(df.loc[df["label"] == 1, "text"]).sum(axis=0).A1

diff = pd.DataFrame({
    "positive": pos,
    "negative": neg,
    "ratio_neg": neg / (pos + 1)
}, index=features)

# Top bigrams surreprésentés dans les négatifs
top_neg = diff.nlargest(15, "ratio_neg")
print("=== Top bigrams → réaction NÉGATIVE ===")
print(top_neg[["negative", "positive", "ratio_neg"]])

# Top bigrams surreprésentés dans les positifs
top_pos = diff.nsmallest(15, "ratio_neg")
print("\n=== Top bigrams → réaction POSITIVE ===")
print(top_pos[["positive", "negative", "ratio_neg"]])   

In [ ]:
print("=== Exemples POSITIFS ===")
for t in df.loc[df["label"] == 0, "text"].sample(5, random_state=42):
    print(f"  • {t[:150]}...\n")

print("=== Exemples NÉGATIFS ===")
for t in df.loc[df["label"] == 1, "text"].sample(5, random_state=42):
    print(f"  • {t[:150]}...\n")   